<a href="https://colab.research.google.com/github/Sucheta-afk/Graphs-for-Emotion-Recognition/blob/main/graph_construction_eeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch-scatter torch-sparse -q -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.4/682.4 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.4 MB/s eta 0:00:00


In [2]:
import torch
from torch_geometric.data import Data
print('PyG imported successfully')
print('Torch version:', torch.__version__)

PyG imported successfully
Torch version: 2.10.0+cpu


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

base = '/content/drive/MyDrive/SEED-IV'

Mounted at /content/drive


In [4]:
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 14),
    'beta':  (14, 31),
    'gamma': (31, 50)
}
from scipy.signal import butter, filtfilt


def bandpass_filter(eeg, lowcut, highcut, fs=200, order=4):
    """
    eeg   : shape (n_channels, n_samples)
    returns filtered eeg of same shape
    """
    nyq = fs / 2
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    # apply filter to each channel independently
    filtered = filtfilt(b, a, eeg, axis=1)
    return filtered
from scipy.signal import hilbert


def compute_plv_matrix(filtered_eeg):
    """
    filtered_eeg : shape (n_channels, n_samples) — already band-pass filtered
    returns      : plv_matrix of shape (n_channels, n_channels)
    """
    n_channels = filtered_eeg.shape[0]

    # Step 1: extract instantaneous phase for every channel
    analytic = hilbert(filtered_eeg, axis=1)       # (62, N_samples) complex
    phase = np.angle(analytic)                      # (62, N_samples) real, in radians

    # Step 2: compute PLV for every channel pair
    plv_matrix = np.zeros((n_channels, n_channels))

    for i in range(n_channels):
        for j in range(i+1, n_channels):  # upper triangle only
            phase_diff = phase[i] - phase[j]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_matrix[i, j] = plv
            plv_matrix[j, i] = plv  # symmetric

    np.fill_diagonal(plv_matrix, 1.0)  # a channel is perfectly locked with itself
    return plv_matrix


def compute_averaged_plv(raw_eeg, fs=200):
    """
    raw_eeg : shape (62, N_samples)
    returns : averaged PLV matrix of shape (62, 62)
    """
    plv_all_bands = []

    for band_name, (low, high) in BANDS.items():
        filtered = bandpass_filter(raw_eeg, low, high, fs=fs)
        plv = compute_plv_matrix(filtered)
        plv_all_bands.append(plv)

    # stack → (5, 62, 62), then average across bands → (62, 62)
    averaged_plv = np.mean(np.stack(plv_all_bands, axis=0), axis=0)
    return averaged_plv

In [5]:
import scipy.io as sio
import numpy as np
import os

raw_path = '/content/drive/MyDrive/SEED-IV/eeg_raw_data/1/'
files = sorted(os.listdir(raw_path))
mat = sio.loadmat(os.path.join(raw_path, files[0]))

raw_eeg = mat['tyc_eeg1']   # replace with your actual key name
print('Raw EEG shape:', raw_eeg.shape)

Raw EEG shape: (62, 33601)


In [6]:
plv = np.load('/content/drive/MyDrive/SEED-IV/plv_test_trial1.npy')
print('PLV shape:', plv.shape)        # (62, 62)
print('Value range:', plv.min().round(3), 'to', plv.max().round(3))

PLV shape: (62, 62)
Value range: 0.033 to 1.0


PyG does not use a (62 × 62) adjacency matrix. It uses edge_index — a (2 × E) tensor where E is the number of edges  
So if channel 3 and channel 7 are connected, there will be two columns: [3, 7] and [7, 3] (both directions, because the graph is undirected (symmetric)).

In [7]:
def plv_to_edge_index(plv_matrix, threshold=0.5):
    """
    plv_matrix : (62, 62) numpy array
    threshold  : keep edges where PLV > threshold
    returns    : edge_index tensor of shape (2, E)
                 edge_weight tensor of shape (E,)
    """
    n = plv_matrix.shape[0]
    source_nodes = []
    target_nodes = []
    edge_weights = []

    for i in range(n):
        for j in range(i+1, n):   # upper triangle only
            if plv_matrix[i, j] > threshold:
                # add both directions (undirected graph)
                source_nodes.extend([i, j])
                target_nodes.extend([j, i])
                edge_weights.extend([plv_matrix[i, j],
                                     plv_matrix[i, j]])

    edge_index = torch.tensor([source_nodes, target_nodes],
                               dtype=torch.long)
    edge_weight = torch.tensor(edge_weights, dtype=torch.float)

    return edge_index, edge_weight

In [8]:
edge_index, edge_weight = plv_to_edge_index(plv, threshold=0.4)
print('edge_index shape:', edge_index.shape)   # (2, E) — E varies
print('edge_weight shape:', edge_weight.shape) # (E,)
print('Number of edges (both directions):', edge_index.shape[1])
print('Unique connections:', edge_index.shape[1] // 2)

edge_index shape: torch.Size([2, 1058])
edge_weight shape: torch.Size([1058])
Number of edges (both directions): 1058
Unique connections: 529


In [9]:
def graph_density(plv_matrix, threshold):
    n = plv_matrix.shape[0]
    max_edges = n * (n - 1) / 2
    actual_edges = np.sum(plv_matrix > threshold) / 2  # upper triangle
    return actual_edges / max_edges

for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    d = graph_density(plv, t)
    n_edges = int(d * 62 * 61 / 2)
    print(f'Threshold {t:.1f} → {n_edges} edges, density {d:.2%}')

Threshold 0.3 → 692 edges, density 36.59%
Threshold 0.4 → 560 edges, density 29.61%
Threshold 0.5 → 407 edges, density 21.52%
Threshold 0.6 → 304 edges, density 16.08%
Threshold 0.7 → 220 edges, density 11.63%


In [12]:
from torch_geometric.data import Data

def build_graph(raw_eeg, de_features, label, subject_id, trial_id, threshold=0.4):
    """
    raw_eeg     : (62, N_samples) — for PLV computation
    de_features : (62, 5) — node features from eeg_feature_smooth
    label       : int — emotion label (0/1/2/3)
    subject_id  : int
    trial_id    : int
    returns     : PyG Data object
    """
    # compute PLV graph
    plv_matrix = compute_averaged_plv(raw_eeg)
    edge_index, edge_weight = plv_to_edge_index(plv_matrix, threshold)

    # node features
    x = torch.tensor(de_features, dtype=torch.float)   # (62, 5)

    # label
    y = torch.tensor([label], dtype=torch.long)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_weight,
        y=y,
        subject_id=subject_id,
        trial_id=trial_id
    )
    return data

In [13]:
dummy_de = np.random.rand(62, 5)   # placeholder until Day 4
label = 0                           # placeholder label

graph = build_graph(raw_eeg, dummy_de, label=0,
                    subject_id=1, trial_id=1)

print(graph)
print('x shape:', graph.x.shape)             # (62, 5)
print('edge_index shape:', graph.edge_index.shape)  # (2, E)
print('y:', graph.y)                          # tensor([0])
print('subject_id:', graph.subject_id)

Data(x=[62, 5], edge_index=[2, 1058], edge_attr=[1058], y=[1], subject_id=1, trial_id=1)
x shape: torch.Size([62, 5])
edge_index shape: torch.Size([2, 1058])
y: tensor([0])
subject_id: 1


All of the functions in a single cell:

In [14]:
def plv_to_edge_index(plv_matrix, threshold=0.5):
    """
    plv_matrix : (62, 62) numpy array
    threshold  : keep edges where PLV > threshold
    returns    : edge_index tensor of shape (2, E)
                 edge_weight tensor of shape (E,)
    """
    n = plv_matrix.shape[0]
    source_nodes = []
    target_nodes = []
    edge_weights = []

    for i in range(n):
        for j in range(i+1, n):   # upper triangle only
            if plv_matrix[i, j] > threshold:
                # add both directions (undirected graph)
                source_nodes.extend([i, j])
                target_nodes.extend([j, i])
                edge_weights.extend([plv_matrix[i, j],
                                     plv_matrix[i, j]])

    edge_index = torch.tensor([source_nodes, target_nodes],
                               dtype=torch.long)
    edge_weight = torch.tensor(edge_weights, dtype=torch.float)

    return edge_index, edge_weight
from torch_geometric.data import Data

def build_graph(raw_eeg, de_features, label, subject_id, trial_id, threshold=0.4):
    """
    raw_eeg     : (62, N_samples) — for PLV computation
    de_features : (62, 5) — node features from eeg_feature_smooth
    label       : int — emotion label (0/1/2/3)
    subject_id  : int
    trial_id    : int
    returns     : PyG Data object
    """
    # compute PLV graph
    plv_matrix = compute_averaged_plv(raw_eeg)
    edge_index, edge_weight = plv_to_edge_index(plv_matrix, threshold)

    # node features
    x = torch.tensor(de_features, dtype=torch.float)   # (62, 5)

    # label
    y = torch.tensor([label], dtype=torch.long)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_weight,
        y=y,
        subject_id=subject_id,
        trial_id=trial_id
    )
    return data
def graph_density(plv_matrix, threshold):
    n = plv_matrix.shape[0]
    max_edges = n * (n - 1) / 2
    actual_edges = np.sum(plv_matrix > threshold) / 2  # upper triangle
    return actual_edges / max_edges

In [15]:
import pickle

save_path = '/content/drive/MyDrive/SEED-IV/test_graph_trial1.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(graph, f)
print('Saved test graph.')

Saved test graph.


#### Day 3 Report:
Graph Construction  

**NODE:**  
1 channel = 1 node  
1 node contains 5 vectors (Bands - tells how much activity is happening at a particular location in the brain)

**EDGES:**  
High PLV(i.e, $> 0.4$ (here)) -> Edges connect

**Conclusion:**  
Connections with a PLV $> 0.4$ are kept as "Edges," while noisy connections are discarded. This resulted in a graph density of approx. 29% (1,058 unique edges), ensuring the model focuses on the most significant neural communication.  